In [72]:
import pandas as pd
import numpy as np 
import duckdb 

In [73]:
df_ts = pd.read_csv("../data/processed/time_series.csv")
df_w = pd.read_csv("../data/processed/weather.csv")

In [74]:
# df_ts.loc[len(df_ts)] = ['15960','2025-05-03','DBRG','2.0']

In [75]:
df_ts = df_ts.rename(columns={"Train": "train_no", "Date": "date", "Station": "station", "Delay": "delay_minutes"})
df_ts["date"] = pd.to_datetime(df_ts["date"])
df_w["date"] = pd.to_datetime(df_w["date"])

In [76]:
df_ts.head()

,train_no,date,station,delay_minutes
0,15960,2025-04-28,DBRG,1.0
1,15960,2025-04-29,DBRG,1.0
2,15960,2025-04-30,DBRG,1.0
3,15960,2025-05-02,DBRG,1.0
4,15960,2025-05-03,DBRG,0.0


In [77]:
df_ts.shape

(120801, 4)

In [79]:
'''[ASSERT] Every date in train data exists in weather data, and vice versa'''

def assert_dates(df_ts,df_w):
    ts_dates = set(df_ts['date'].dt.date)
    w_dates = set(df_w['date'].dt.date)
    #set difference
    missing_in_weather = ts_dates - w_dates
    missing_in_train = w_dates - ts_dates

    assert not missing_in_weather and not missing_in_train,(
        f"Mismatch!\n"
        f"Missing in weather: {missing_in_weather}\n"
        f"Extra in weather: {missing_in_train}"
    )
    
assert_dates(df_ts,df_w)


In [80]:
print(df_w.shape)
print(df_ts.shape)

(3281, 11)
(120801, 4)


In [24]:
df_w.sample(5)

,train_no,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
986,15615,2026-01-12,24.6,13.8,18.5,0.0,0.0,7.1,20.9,80,0
1724,12423,2026-01-20,24.0,11.9,17.6,0.0,0.0,6.2,13.7,74,2
303,15960,2026-02-25,27.9,16.1,21.4,1.5,1.5,15.1,25.6,66,55
677,12067,2026-03-08,31.5,19.7,25.5,5.2,5.2,17.9,38.5,67,63
2504,15657,2026-03-11,36.7,21.2,29.1,0.0,0.0,16.3,43.2,35,3


In [6]:
df_ts.info()

<class 'pandas.DataFrame'>
RangeIndex: 120801 entries, 0 to 120800
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   train_no       120801 non-null  int64         
 1   date           120801 non-null  datetime64[us]
 2   station        120801 non-null  str           
 3   delay_minutes  108319 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 3.7 MB


In [26]:
df = df_ts.merge(df_w,on=['date','train_no'],how='inner')
df.head()

,train_no,date,station,delay_minutes,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
0,15960,2025-04-28,DBRG,1.0,26.8,19.5,23.0,4.3,4.3,13.8,36.4,84,61
1,15960,2025-04-29,DBRG,1.0,27.5,19.0,23.3,0.4,0.4,8.3,18.4,81,51
2,15960,2025-04-30,DBRG,1.0,30.6,20.0,25.7,0.0,0.0,6.5,16.2,71,3
3,15960,2025-05-02,DBRG,1.0,28.5,20.7,24.7,1.3,1.3,15.9,34.6,76,61
4,15960,2025-05-03,DBRG,0.0,29.4,20.7,24.7,7.7,7.7,6.4,18.7,82,63


In [27]:
df.sample(5)

,train_no,date,station,delay_minutes,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
32528,12346,2025-11-15,GHY,2.0,28.0,17.9,22.4,0.0,0.0,8.7,22.7,76,0
92253,15909,2026-02-08,RNY,57.0,25.2,12.8,18.6,0.0,0.0,10.1,23.4,69,1
61189,15657,2025-12-31,SGG,425.0,18.9,8.3,12.7,0.0,0.0,9.1,24.1,87,3
49248,15946,2025-07-13,HJI,11.0,36.9,27.5,32.0,0.0,0.0,9.3,21.2,73,3
30364,15615,2025-12-11,HLX,184.0,24.8,14.1,18.8,0.0,0.0,10.0,24.8,81,3


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120801 entries, 0 to 120800
Data columns (total 13 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   train_no                   120801 non-null  int64         
 1   date                       120801 non-null  datetime64[us]
 2   station                    120801 non-null  str           
 3   delay_minutes              108319 non-null  float64       
 4   temperature_2m_max         120801 non-null  float64       
 5   temperature_2m_min         120801 non-null  float64       
 6   temperature_2m_mean        120801 non-null  float64       
 7   precipitation_sum          120801 non-null  float64       
 8   rain_sum                   120801 non-null  float64       
 9   wind_speed_10m_max         120801 non-null  float64       
 10  wind_gusts_10m_max         120801 non-null  float64       
 11  relative_humidity_2m_mean  120801 non-null  int64         
 12 

In [29]:
print(f"{df.shape} {df_ts.shape} {df_w.shape}") 

(120801, 13) (120801, 4) (3281, 11)


In [87]:

''' [ASSERT] a lossless merge of two dataset '''
def assert_lossless_join(df_merged,df_ts):
    assert df_merged.shape[0] ==  df_ts.shape[0]
'''[ASSERT] weather columns are not null post-join '''
def assert_weather_col(df_merged,df_w):
    df_weather_col = df_w.drop(columns=['date','train_no']).columns
    assert df_merged[df_weather_col].isna().sum().sum() == 0,(
        "Mismatch \n",
        f"{df_merged[df_weather_col].isna().sum()}"
    )
assert_weather_col(df,df_w)

In [8]:
''' [CAUTION] each row is a station info not a single day

train_no | date       | station
15960    | 2025-04-28 | DBRG
15960    | 2025-04-28 | (another station...)
...
15960    | 2026-04-25 | HWH

each date can have more than 1 station so row number > 365 

'''
df[df['train_no']==15960]

,train_no,date,station,delay_minutes,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
0,15960,2025-04-28,DBRG,1.0,26.8,19.5,23.0,4.3,4.3,13.8,36.4,84,61
1,15960,2025-04-29,DBRG,1.0,27.5,19.0,23.3,0.4,0.4,8.3,18.4,81,51
2,15960,2025-04-30,DBRG,1.0,30.6,20.0,25.7,0.0,0.0,6.5,16.2,71,3
3,15960,2025-05-02,DBRG,1.0,28.5,20.7,24.7,1.3,1.3,15.9,34.6,76,61
4,15960,2025-05-03,DBRG,0.0,29.4,20.7,24.7,7.7,7.7,6.4,18.7,82,63
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15855,15960,2026-04-20,HWH,98.0,27.0,21.2,23.3,32.2,32.2,13.4,33.5,91,65
15856,15960,2026-04-21,HWH,100.0,25.7,20.8,22.2,69.7,69.7,14.9,40.0,95,65
15857,15960,2026-04-22,HWH,-5.0,26.3,20.7,22.7,3.5,3.5,9.1,21.6,89,55
15858,15960,2026-04-24,HWH,-10.0,32.3,22.5,27.7,5.4,5.4,13.3,24.1,72,63


In [9]:
df[df['delay_minutes']<0]

,train_no,date,station,delay_minutes,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
15601,15960,2025-04-29,HWH,-13.0,27.5,19.0,23.3,0.4,0.4,8.3,18.4,81,51
15602,15960,2025-04-30,HWH,-19.0,30.6,20.0,25.7,0.0,0.0,6.5,16.2,71,3
15604,15960,2025-05-03,HWH,-36.0,29.4,20.7,24.7,7.7,7.7,6.4,18.7,82,63
15607,15960,2025-05-07,HWH,-40.0,28.4,21.1,23.5,63.2,63.2,12.4,32.4,88,65
15608,15960,2025-05-09,HWH,-21.0,31.5,22.8,27.0,2.5,2.5,9.9,22.3,79,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...
120746,15909,2026-03-03,LGH,-10.0,27.2,15.2,21.1,0.0,0.0,7.7,20.5,70,1
120755,15909,2026-03-12,LGH,-15.0,25.4,19.4,21.9,3.3,3.3,11.7,27.7,83,61
120766,15909,2026-03-23,LGH,-10.0,24.0,18.1,20.3,5.6,5.6,8.4,21.6,85,61
120791,15909,2026-04-17,LGH,-12.0,30.7,20.8,25.9,0.0,0.0,7.0,15.8,74,2


In [10]:
#[WARN]
'''0.0 -> no delay
NULL -> missing / not recorded'''
mising_pct = df.delay_minutes.isnull().sum()/df.shape[0] * 100
# print(type(mising_pct))
print(f"{np.round(mising_pct,2)}%")

10.33%


In [11]:
df.shape

(120801, 13)

In [12]:
#How much data do we have 
result = duckdb.sql(
    """select
    train_no,
    count(*) as total_records,
    MIN(date) as earliest_date,
    MAX(date) as latest_date,
    COUNT(distinct date ) as days_covered
    from df
    group by train_no
    order by days_covered
    """  
).to_df()
result

,train_no,total_records,earliest_date,latest_date,days_covered
0,15946,5985,2025-04-27,2026-04-26,105
1,15960,15860,2025-04-28,2026-04-25,260
2,12067,4056,2025-04-28,2026-04-25,312
3,12423,9490,2025-04-27,2026-04-26,365
4,12346,6570,2025-04-27,2026-04-26,365
5,15909,36135,2025-04-27,2026-04-26,365
6,12505,12775,2025-04-27,2026-04-26,365
7,15615,12410,2025-04-27,2026-04-26,365
8,15657,17520,2025-04-27,2026-04-26,365


In [13]:
#check for nulls in delays per train
result = duckdb.sql("""
select
    train_no,
    Count(*) total_rows,
    Count(delay_minutes) as non_null_delays,
    Count(*) - Count(delay_minutes) as missing_count,
    Round(
        (Count(*)-Count(delay_minutes))/Count(*)*100.0
        ,2) as missing_pct
    from df 
    group by train_no,
    order by missing_pct
                    """).to_df()
result

,train_no,total_rows,non_null_delays,missing_count,missing_pct
0,15946,5985,5801,184,3.07
1,15960,15860,14767,1093,6.89
2,15657,17520,16259,1261,7.20
3,12423,9490,8731,759,8.00
4,12346,6570,5950,620,9.44
5,15909,36135,31724,4411,12.21
6,15615,12410,10878,1532,12.34
7,12067,4056,3509,547,13.49
8,12505,12775,10700,2075,16.24


In [14]:
''' 
[WARN] how to better reprenst delay freq if some train have more missing data then other train as 
total missing is 10% individual trains pct differ
(Minimal sol) Create a weighted score
    high delay + high data -> high score
    high delay + low data -> reduced score
'''
#3.most delayed trains ranking 

result = duckdb.sql("""
    select
        train_no,
        Round(AVG(delay_minutes),2) as avg_delay,
        MAX(delay_minutes) as worst_delay,
        Count(Distinct date) as total_days_running,
        -- counting distinct days when delay > 10min
        Count(
            Distinct Case When delay_minutes>10 then date end)as days_delayed,
        ROUND(
        (COUNT(DISTINCT CASE WHEN delay_minutes > 10 THEN date END) * 1.0
        / COUNT(DISTINCT CASE WHEN delay_minutes IS NOT NULL THEN date END))
        *
        (COUNT(DISTINCT CASE WHEN delay_minutes IS NOT NULL THEN date END) * 1.0
        / COUNT(DISTINCT date))
        * 100,
    2) AS adjusted_delay_score_freq
    from df,
    group by train_no 
    order by adjusted_delay_score_freq DESC , avg_delay DESC
""").to_df()
result

,train_no,avg_delay,worst_delay,total_days_running,days_delayed,adjusted_delay_score_freq
0,15657,77.39,744.0,365,365,100.00
1,15946,41.10,703.0,105,105,100.00
2,15960,38.25,963.0,260,260,100.00
3,12423,31.89,696.0,365,365,100.00
4,12346,29.84,503.0,365,362,99.18
5,15615,63.97,2949.0,365,354,96.99
6,15909,76.91,970.0,365,352,96.44
7,12505,39.85,732.0,365,340,93.15
8,12067,9.16,97.0,312,266,85.26


In [15]:
# Monthly delay trend
result = duckdb.sql("""
    select
        date_trunc('month',date) as month,
        Round(AVG(delay_minutes),2) as avg_delay,       
        COUNT(DISTINCT train_no) AS trains_running
    FROM df
    WHERE delay_minutes IS NOT NULL
    GROUP BY month
    ORDER BY month;
""").to_df()
result

,month,avg_delay,trains_running
0,2025-04-01,43.98,9
1,2025-05-01,40.86,9
2,2025-06-01,39.02,9
3,2025-07-01,42.74,9
4,2025-08-01,31.72,9
5,2025-09-01,45.87,9
6,2025-10-01,61.34,9
7,2025-11-01,58.75,9
8,2025-12-01,111.73,9
9,2026-01-01,92.43,9


In [16]:
#delay buckets
#without daily_dely we we counting station on same day mor then once 
result = duckdb.sql("""
                    
WITH daily_delay AS (
    SELECT
        train_no,
        date,
        MAX(delay_minutes) AS max_delay,
        MIN(delay_minutes) AS min_delay
    FROM df
    WHERE delay_minutes IS NOT NULL
    GROUP BY train_no, date
)
SELECT
    train_no,
    COUNT(CASE WHEN min_delay < 0 THEN 1 END) AS early_days,
    COUNT(CASE WHEN max_delay <= 10 THEN 1 END) AS on_time,
    COUNT(CASE WHEN max_delay > 10 AND max_delay <= 30 THEN 1 END) AS minor_delay,
    COUNT(CASE WHEN max_delay > 30 AND max_delay <= 120 THEN 1 END) AS moderate_delay,
    COUNT(CASE WHEN max_delay > 120 THEN 1 END) AS severe_delay

FROM daily_delay
GROUP BY train_no
ORDER BY severe_delay DESC;
 """).to_df()
result

,train_no,early_days,on_time,minor_delay,moderate_delay,severe_delay
0,15909,51,0,0,121,231
1,15657,236,0,8,249,108
2,12423,83,0,56,238,71
3,15615,115,11,4,282,68
4,12505,100,0,1,273,66
5,12346,104,3,143,171,48
6,15960,157,0,2,215,43
7,15946,19,0,0,74,31
8,12067,237,45,205,61,0
